### Sankey diagrams for a country model
For illustration, exemplary report data of a Norway prototype can be used.

In [1]:
# Create default report (can be skipped to reproduce Norway example)

#from ixmp import Platform
#from message_ix import Scenario
#from message_ix.reporting import Reporter

#model = 'MESSAGE_NO'
#scen = 'update' 

#mp = Platform()
#scen = Scenario(mp, model, scen)
#rep = Reporter.from_scenario(scen)
#df = rep.get('message:default') # in fact, only in = input x ACT, out = output x ACT is needed

In [2]:
import pandas as pd
import numpy as np
import pyam

import plotly.graph_objects as go

In [3]:
# Use exemplary report

report = pd.read_csv('MESSAGE_NO_example_default_report.csv')
report.Unit.fillna('', inplace=True)
df = pyam.IamDataFrame(report)

C:\Users\brend\AppData\Local\Temp\ipykernel_37408\60054878.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  report.Unit.fillna('', inplace=True)
C:\Users\brend\AppData\Local\Temp\ipykernel_37408\60054878.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  report.Unit.fillna('', inplace=True)
c:\Users\brend\anaconda3\envs\message_env\Lib\site-packages\

In [4]:
# Plotted years

years = list(range(2020,2060,5)) + list(range(2060,2120,10))

In [5]:
# Plotted region (generalization for multi-region models; not mandatory for the example)

region = 'Norway' # choose region
df.filter(region = region+'*').timeseries()

1960  1965  \
model      scenario region       variable                 unit               
MESSAGE_NO update   Norway       capacity|LNG_exp                NaN   NaN   
                                 capacity|adipic_thermal         NaN   NaN   
                                 capacity|ammonia_secloop        NaN   NaN   
                                 capacity|bio_hpl                NaN   NaN   
                                 capacity|bio_istig              NaN   NaN   
...                                                              ...   ...   
                    Norway|World out|stocks|puq|nuc_lc|M1        NaN   NaN   
                                 out|stocks|u5q|nuc_hc|M1        NaN   NaN   
                                 out|stocks|u5q|nuc_lc|M1        NaN   NaN   
                                 out|stocks|uq|nuc_hc|M1         NaN   NaN   
                                 out|stocks|uq|nuc_lc|M1         NaN   NaN   

                                                                1970  1975  \
model      scenario region       variable                 unit               
MESSAGE_NO update   Norway       capacity|LNG_exp                NaN   NaN   
                                 capacity|adipic_thermal         NaN   NaN   
                                 capacity|ammonia_secloop        NaN   NaN   
                                 capacity|bio_hpl                NaN   NaN   
                                 capacity|bio_istig              NaN   NaN   
...                                                              ...   ...   
                    Norway|World out|stocks|puq|nuc_lc|M1        NaN   NaN   
                                 out|stocks|u5q|nuc_hc|M1        NaN   NaN   
                                 out|stocks|u5q|nuc_lc|M1        NaN   NaN   
                                 out|stocks|uq|nuc_hc|M1         NaN   NaN   
                                 out|stocks|uq|nuc_lc|M1         NaN   NaN   

                                                                1980  1985  \
model      scenario region       variable                 unit               
MESSAGE_NO update   Norway       capacity|LNG_exp                NaN   NaN   
                                 capacity|adipic_thermal         NaN   NaN   
                                 capacity|ammonia_secloop        NaN   NaN   
                                 capacity|bio_hpl                NaN   NaN   
                                 capacity|bio_istig              NaN   NaN   
...                                                              ...   ...   
                    Norway|World out|stocks|puq|nuc_lc|M1        NaN   NaN   
                                 out|stocks|u5q|nuc_hc|M1        NaN   NaN   
                                 out|stocks|u5q|nuc_lc|M1        NaN   NaN   
                                 out|stocks|uq|nuc_hc|M1         NaN   NaN   
                                 out|stocks|uq|nuc_lc|M1         NaN   NaN   

                                                                1990  1995  \
model      scenario region       variable                 unit               
MESSAGE_NO update   Norway       capacity|LNG_exp                NaN   NaN   
                                 capacity|adipic_thermal         NaN   NaN   
                                 capacity|ammonia_secloop        NaN   NaN   
                                 capacity|bio_hpl                0.0   0.0   
                                 capacity|bio_istig              NaN   NaN   
...                                                              ...   ...   
                    Norway|World out|stocks|puq|nuc_lc|M1        0.0   0.0   
                                 out|stocks|u5q|nuc_hc|M1        0.0   0.0   
                                 out|stocks|u5q|nuc_lc|M1        0.0   0.0   
                                 out|stocks|uq|nuc_hc|M1         0.0   0.0   
                                 out|stocks|uq|nuc_lc|M1         0.0   0.0   

    

In [6]:
# Create data for Sankey diagrams

df.filter(year=years, variable=['in|*', 'out|*'], inplace=True)
df = df.rename(unit={'': 'GW'}).convert_unit('GW', 'TWh/yr')
all_flows = df.timeseries().reset_index()

# Split the strings in the identified variables for further processing
splitted_vars = [v.split('|') for v in all_flows.variable]

# Create auxilary dataframes for processing
aux1_df = pd.DataFrame(splitted_vars, columns=['flow_type', 'level', 'commodity', 'technology', 'mode']) 
aux2_df = pd.concat([all_flows.reset_index(drop=True), aux1_df.reset_index(drop=True)], axis=1)

In [10]:
# The following loop includes two simultaneous steps: 
# 1) the creation of csv file which can be used in other Sankey tools (e.g.: https://datavis.indecol.no/sankey)
# 2) plotting Sankey diagrams using plotly

for year in years: 
    csv = pd.DataFrame()
    Source = []
    Target = []
    Value = []
    for i, j in aux2_df.iterrows():
        if j[['flow_type']].item() == 'in':
            source = [j[['level']].item()+'_'+j[['commodity']].item()]
            target = [j[['technology']].item()+'_'+j[['mode']].item()]
            value = [j[[year]].item()]
        elif j[['flow_type']].item() == 'out':
            source = [j[['technology']].item()+'_'+j[['mode']].item()]
            target = [j[['level']].item()+'_'+j[['commodity']].item()]
            value = [j[[year]].item()]
        Source = Source + source
        Target = Target + target
        Value = Value + value
    csv['Source'] = Source
    csv['Target'] = Target
    csv['Value'] = Value
    
    # Drop flows which are 0 and infintesimal
    csv = csv[csv.Value != 0.0]
    csv = csv[csv.Value >= 0.0001]
    
    # Drop not necessary flows (certainly depends on the model)
    csv = csv[csv.Target != 'emi_dummy_emi_dummy']   
    csv = csv[csv.Source != 'oil_extr_mpen_M1']
    csv = csv[csv.Source != 'gas_extr_mpen_M1']
    csv = csv[csv.Source != 'cement_CO2_M1']
    csv = csv[csv.Source != 'coal_extr_mpen_M1']
    csv = csv[csv.Source != 'flaring_CO2_M1']
    csv = csv[csv.Source != 'co2_tr_dis_M1']
    
    # End of STEP 1 (write csv file and use it e.g. on https://datavis.indecol.no/sankey
    #csv.to_csv('SANKEY_%s_%s.csv' %(region, year), header=None, index=False)
    
    # Start of STEP 2 using plotly
    # Since plotly uses integers instead of strings to define the source and the target of a flow,
    # the strings of csv need to be converted.

    # Identify an unique integer for the strings of csv (labeling)
    # Concatenate the 'Source' and 'Target' columns and get unique values
    l = pd.DataFrame(pd.concat([csv['Source'], csv['Target']]).unique(), columns=['Combined'])
    l.reset_index(level=0, inplace=True)
    #l.set_index(0, inplace=True)

    # Assign integers to the strings in source column
    s = pd.DataFrame(csv['Source'])
    s.set_index('Source', drop=True, inplace=True)
    s.index.names = [0]
    s['index']='nan'
    s.update(l)

    # Assign integers to the strings in target column
    t = pd.DataFrame(csv['Target'])
    t.set_index('Target', drop=True, inplace=True)
    t.index.names = [0]
    t['index']='nan'
    t.update(l)

    # Create sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node = dict(
            pad = 15,
            thickness = 10,
            line = dict(color = "black", width = 0.5),
            label = list(l.index), # use the index of the labeled strings as integers
            hovertemplate='%{label}: %{value} TWh<extra></extra>',  
            color = "blue"
        ),
        link = dict(
            source = list(s['index']), # each integer corresponds to the index of the labeled strings
            target = list(t['index']), # each integer corresponds to the index of the labeled strings
            value = list(csv['Value']),
            hovertemplate='"%{source.label}" to "%{target.label}": %{value} TWh<extra></extra>'
        ))])

    fig.update_layout(title_text="%s_%s" %  (region, year), font_size=10)
    fig.show()
    
    # Write to html
    #fig.write_html("%s_%s.html" % (region, year))

In [9]:
l

,index,Combined
0,0,final_biomass
1,1,final_d_heat
2,2,final_electr
3,3,final_ethanol
4,4,final_fueloil
...,...,...
107,107,export_ethanol
108,108,export_fueloil
109,109,export_gas
110,110,export_lightoil


In [ ]:
#mp.close_db()